In [56]:
from datasets import get_dataset_config_names, get_dataset_split_names

dataset_id = "AyoubChLin/Company-document-dataset-v2"
config_names = get_dataset_config_names(dataset_id)

print(config_names)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

['all', 'account_statement', 'credit_note', 'employment_certificate', 'goods_received_note', 'inventory_report', 'invoice', 'packing_slip', 'payslip', 'purchase_order', 'quote', 'receipt', 'shipping_order', 'work_order', 'scans']


In [57]:
from datasets import load_dataset, Pdf

stream = load_dataset(dataset_id, "all", split="train", streaming=True)
stream = stream.cast_column("pdf", Pdf(decode = False))

print(stream.features)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

{'pdf': Pdf(decode=False), 'file_name': Value('string'), 'doc_id': Value('string'), 'split': Value('string'), 'document_type': Value('string'), 'source': Value('string'), 'variant': Value('string'), 'company': Value('string'), 'company_name': Value('string'), 'company_sector': Value('string'), 'company_country': Value('string'), 'company_city': Value('string'), 'company_tax_id': Value('string'), 'company_locale': Value('string'), 'company_currency': Value('string'), 'number': Value('string'), 'issue_date': Value('string'), 'currency': Value('string'), 'total': Value('string'), 'items_count': Value('int32'), 'counterparty_role': Value('string'), 'counterparty_name': Value('string'), 'layout': Value('string'), 'theme': Value('string'), 'locale': Value('string'), 'pages': Value('int32'), 'extracted_data': Value('string'), 'file_content': Value('string'), 'options': Value('string'), 'rendered': Value('string'), 'boxes': Value('string'), 'words': Value('string')}


In [58]:
splits = get_dataset_split_names(dataset_id, config_name="all")
print(splits)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

['train', 'validation', 'test']


In [59]:
first_record = next(iter(stream))
print({k: v for k, v in first_record.items() if k != "pdf"})

{'file_name': 'account_statement/fa214a0ee7a32890.pdf', 'doc_id': 'fa214a0ee7a32890', 'split': 'train', 'document_type': 'account_statement', 'source': 'adventureworks', 'variant': 'account_statement.adventureworks', 'company': 'vantage-outdoor-corp', 'company_name': 'Vantage Outdoor Corp.', 'company_sector': 'bikes', 'company_country': 'USA', 'company_city': 'Portland', 'company_tax_id': 'EIN 43-3439968', 'company_locale': 'en', 'company_currency': 'USD', 'number': 'ST-2023-29912', 'issue_date': '2023-02-23', 'currency': 'USD', 'total': '27052.20', 'items_count': 3, 'counterparty_role': 'customer', 'counterparty_name': 'Daring Rides', 'layout': 'modern', 'theme': 'fresh', 'locale': 'en', 'pages': 2, 'extracted_data': '{"doc_type": "account_statement", "number": "ST-2023-29912", "issue_date": "2023-02-23", "currency": "USD", "parties": {"issuer": {"name": "Vantage Outdoor Corp.", "address": {"line1": "304 5th Avenue", "line2": "Suite 542", "city": "Portland", "region": "OR", "postal_co

In [60]:
from collections import Counter
from tqdm.auto import tqdm

train_scan = load_dataset(dataset_id, "all", split="train", streaming=True, columns = ["doc_id", "document_type"])
type_counts = Counter()
record_scanned = 0

for record in tqdm(train_scan, desc = "Scanning training records"):
    record_scanned += 1
    type_counts[record["document_type"]] += 1

print(f"Records scanned: {record_scanned: ,}")

for document_type, count in sorted(type_counts.items()):
    print(f"{document_type}: {count: ,}")

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Scanning training records: 0it [00:00, ?it/s]

Records scanned:  259,497
account_statement:  522
credit_note:  35,107
employment_certificate:  202
goods_received_note:  2,667
inventory_report:  35
invoice:  35,342
packing_slip:  35,040
payslip:  231
purchase_order:  36,908
quote:  35,004
receipt:  12,105
shipping_order:  35,182
work_order:  31,152


In [61]:
import pandas as pd

counts_df = pd.DataFrame(sorted(type_counts.items()), columns = ["document_type", "train_count"])
counts_df.to_csv("type_counts_train.csv", index = False)
counts_df

,document_type,train_count
0,account_statement,522
1,credit_note,35107
2,employment_certificate,202
3,goods_received_note,2667
4,inventory_report,35
5,invoice,35342
6,packing_slip,35040
7,payslip,231
8,purchase_order,36908
9,quote,35004


In [62]:
from datasets import load_dataset_builder
rows = []
for t in [c for c in config_names if c not in ("all", "scans")]:
    splits = load_dataset_builder(dataset_id, t).info.splits or {}
    rows.append({"document_type": t, **{s: v.num_examples for s, v in splits.items()}})
counts_all = pd.DataFrame(rows)
counts_all.to_csv("type_counts_all_splits.csv", index=False)

check = counts_all.merge(counts_df, on="document_type")
print("train counts match the scan:", (check["train"] == check["train_count"]).all())
counts_all

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

train counts match the scan: True


,document_type,train,validation,test
0,account_statement,522,99,107
1,credit_note,35107,6337,6303
2,employment_certificate,202,43,45
3,goods_received_note,2667,512,510
4,inventory_report,35,6,10
5,invoice,35342,6446,6370
6,packing_slip,35040,6456,6244
7,payslip,231,30,29
8,purchase_order,36908,6598,6807
9,quote,35004,6313,6429


In [63]:
import json 

record_data = json.loads(first_record["extracted_data"])
print(json.dumps(record_data, indent = 2, ensure_ascii = False))

{
  "doc_type": "account_statement",
  "number": "ST-2023-29912",
  "issue_date": "2023-02-23",
  "currency": "USD",
  "parties": {
    "issuer": {
      "name": "Vantage Outdoor Corp.",
      "address": {
        "line1": "304 5th Avenue",
        "line2": "Suite 542",
        "city": "Portland",
        "region": "OR",
        "postal_code": "97205",
        "country": "USA"
      },
      "phone": "(442) 555-7651",
      "email": "sales@vantageoutdoor.com",
      "website": "www.vantageoutdoor.com",
      "tax_id": "EIN 43-3439968",
      "code": "US",
      "bank": {
        "bank_name": "Crédit Régional",
        "account": "5113587454",
        "routing": "857881569"
      }
    },
    "customer": {
      "name": "Daring Rides",
      "address": {
        "line1": "Bay Area Outlet Mall",
        "city": "Clearwater",
        "region": "FL",
        "postal_code": "33755",
        "country": "United States"
      },
      "contact": "Russell King",
      "code": "AW00029912"
    }

In [64]:
print(json.dumps(record_data.get("items", []), indent = 2, ensure_ascii = False))

[
  {
    "description": "Invoice VO-22044309",
    "quantity": 1,
    "unit_price": "0.00",
    "discount_rate": "0",
    "tax_rate": "0",
    "line_total": "0.00",
    "extra": {
      "date": "2022-08-30",
      "debit": "31091.81",
      "balance": "31091.81"
    }
  },
  {
    "description": "Payment received, thank you · VO-22044309",
    "quantity": 1,
    "unit_price": "0.00",
    "discount_rate": "0",
    "tax_rate": "0",
    "line_total": "0.00",
    "extra": {
      "date": "2022-10-09",
      "credit": "31091.81",
      "balance": "0.00"
    }
  },
  {
    "description": "Invoice VO-22045065",
    "quantity": 1,
    "unit_price": "0.00",
    "discount_rate": "0",
    "tax_rate": "0",
    "line_total": "0.00",
    "extra": {
      "date": "2022-11-30",
      "debit": "27052.20",
      "balance": "27052.20"
    }
  }
]


In [65]:
target_number = "VO-22044309"
invoice_stream = load_dataset(dataset_id, "invoice", split="train", streaming=True, columns = ["doc_id", "company_name", "counterparty_name", "number", "issue_date", "currency", "total"])

matches = []

for invoice in tqdm(invoice_stream, desc = "searching training invoices"):
    if invoice["number"] == target_number:
        matches.append(invoice)

print (f"matches found: {len(matches)}")
for match in matches:
    print(match)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

searching training invoices: 0it [00:00, ?it/s]

matches found: 0


In [66]:
other_split_matches = []

for split_name in ["validation", "test"]:
    invoice_stream = load_dataset(dataset_id, "invoice", split=split_name, streaming=True, columns = ["doc_id", "company_name", "counterparty_name", "number", "issue_date", "currency", "total"])
    
    for invoice in tqdm(invoice_stream, desc = f"searching {split_name} invoices"):
        if invoice["number"] == target_number:
            other_split_matches.append({"split": split_name, **invoice})

print(f"matches found in validation/test: {len(other_split_matches)}")
for match in other_split_matches:
    print(match)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

searching validation invoices: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

searching test invoices: 0it [00:00, ?it/s]

matches found in validation/test: 0


In [67]:
import itertools, collections

types = [c for c in config_names if c not in ("all", "scans")]
cols = ["doc_id", "company_name", "counterparty_name", "number", "issue_date", "currency", "total", "extracted_data"]

samples, ref_keys = {}, {}
for t in types:
    ds = load_dataset(dataset_id, t, split="train", streaming=True, columns = cols)
    rows = list(itertools.islice(ds, 200))
    samples[t] = rows
    c = collections.Counter()
    for r in rows:
        c.update(json.loads(r["extracted_data"]).get("refs", {}).keys())
    ref_keys[t] = c
    print(f"\n=== {t} === refs keys: {dict(c)}")
    print(f" example: ", json.loads(rows[0]["extracted_data"]).get("refs"))

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]


=== account_statement === refs keys: {'account_number': 200, 'period': 200}
 example:  {'account_number': 'AW00029912', 'period': '28 Aug 2022 – 23 Feb 2023'}


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

...


=== shipping_order === refs keys: {'order_id': 200, 'tracking_number': 32}
 example:  {'order_id': 'SO71568'}


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]


=== work_order === refs keys: {}
 example:  None


In [68]:
def paths(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from paths(v, f"{prefix}{k}.")
    elif isinstance(obj, list) and obj:
        yield from paths(obj[0], f"{prefix}[].")
    else:
        yield prefix.rstrip(".")

schema_rows = []
for t, rows in samples.items():
    c = collections.Counter(p for r in rows for p in set(paths(json.loads(r["extracted_data"]))))
    schema_rows += [{"document_type": t, "field": p, "present_in_200": n} for p, n in c.items()]
schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv("schema_per_type.csv", index=False)

for t, g in schema_df.groupby("document_type"):
    print(f"\n== {t} ({len(g)} fields) ==")
    print(g.sort_values("field").to_string(index=False))


== account_statement (47 fields) ==
    document_type                                field  present_in_200
account_statement                             currency             200
account_statement                             doc_type             200
account_statement                    extra.aging.[].[]             141
account_statement                           issue_date             200
account_statement                 items.[].description             200
account_statement               items.[].discount_rate             200
account_statement               items.[].extra.balance             200
account_statement                items.[].extra.credit              35
account_statement                  items.[].extra.date             200
account_statement                 items.[].extra.debit             165
account_statement                  items.[].line_total             200
account_statement                    items.[].quantity             200
account_statement                    ite

In [69]:
def light_scan(t, split = "train"):
    ds = load_dataset(dataset_id, t, split=split, streaming=True, columns = ["doc_id", "company_name", "counterparty_name", "number", "issue_date", "total", "extracted_data"])
    for r in ds:
        d = json.loads(r["extracted_data"])
        yield {
            "doc_id": r["doc_id"], "type": t, "company_name": r["company_name"], 
            "counterparty_name": r["counterparty_name"], "number": r["number"], "issue_date": r["issue_date"],
            "total": r["total"], "refs": json.dumps(d.get("refs") or {}, ensure_ascii=False),
            "items": json.dumps(
                [(i.get("description"), i.get("quantity")) for i in d.get("items", [])],
                ensure_ascii=False)
        }

for t in ["invoice", "credit_note"]:
    df = pd.DataFrame(tqdm(light_scan(t), desc = t))
    df.to_parquet(f"index_{t}.parquet")

invoice: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

credit_note: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [70]:
inv = pd.read_parquet("index_invoice.parquet")
cn = pd.read_parquet("index_credit_note.parquet")
ref_col = cn["refs"].apply(lambda s: json.loads(s).get("original_invoice"))
resolved = ref_col.dropna().isin(set(inv["number"]))
print(f"{resolved.sum(): ,} of {ref_col.notna().sum(): ,} credit notes point to an existing invoice")

 4,623 of  35,107 credit notes point to an existing invoice


In [71]:
purchase_orders = pd.DataFrame(tqdm(light_scan("purchase_order"), desc = "Indexing purchase orders"))
purchase_orders.to_parquet("index_purchase_order.parquet", index = False)
print(f"Purchase orders indexed: {len(purchase_orders): ,}")

Indexing purchase orders: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Purchase orders indexed:  36,908


In [72]:
invoice_po_refs = inv["refs"].apply(lambda s: json.loads(s).get("po_number"))

purchase_order_numbers = set(purchase_orders["number"].dropna())

has_po_ref = invoice_po_refs.notna()
matching_po = invoice_po_refs[has_po_ref].isin(purchase_order_numbers)

print(f"Invoices with a PO reference: {has_po_ref.sum(): ,}")
print(f"PO references matching an existing order: {matching_po.sum(): ,}")

Invoices with a PO reference:  14,171
PO references matching an existing order:  0


In [73]:
print("Invoice PO references:")
print(invoice_po_refs.dropna().head(10).tolist())

print("\nPurchase order numbers:")
print(purchase_orders["number"].dropna().head(10).tolist())

Invoice PO references:
['PO-59244', 'PO16008173883', 'PO-84053', 'PO2233118021', 'PO-7663', 'PO-39836', 'PO-75204', 'PO-47778', 'PO-3862', 'PO-51505']

Purchase order numbers:
['VO-PO-249', 'PO-2025-1765', 'VO-PO-830', 'PO-2025-2262', 'PO002077', 'KC-PO-100', 'PO-2025-3966', 'PO002594', 'PO-2025-2895', 'IS-PO-3615']


In [74]:
overlap = set(invoice_po_refs.dropna()) & purchase_order_numbers

print(f"Exact matches: {len(overlap): ,}")
print(list(overlap)[:10])

Exact matches:  0
[]


In [75]:
goods_received_notes = pd.DataFrame(tqdm(light_scan("goods_received_note"), desc = "Indexing goods-received notes"))

goods_received_notes.to_parquet("index_goods_received_note.parquet", index = False)

grn_po_refs = goods_received_notes["refs"].apply(lambda s: json.loads(s).get("po_reference"))

has_po_ref = grn_po_refs.notna()
matching_po = grn_po_refs[has_po_ref].isin(purchase_order_numbers)

print(f"Goods-received note indexes: {len(goods_received_notes): ,}")
print(f"Notes with a PO reference: {has_po_ref.sum(): ,}")
print(f"References matching a purchase order: {matching_po.sum(): ,}")

Indexing goods-received notes: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Goods-received note indexes:  2,667
Notes with a PO reference:  2,667
References matching a purchase order:  476


In [76]:
goods_received_notes["po_reference"] = goods_received_notes["refs"].apply(lambda s: json.loads(s).get("po_reference"))

matched_pairs = goods_received_notes.merge(purchase_orders, left_on = "po_reference", right_on = "number", suffixes = ("_grn", "_po"))

display(matched_pairs[["po_reference", "company_name_grn", "counterparty_name_grn", "company_name_po", "counterparty_name_po"]].head(5))

,po_reference,company_name_grn,counterparty_name_grn,company_name_po,counterparty_name_po
0,VO-PO-3378,Vantage Outdoor Corp.,"Litware, Inc.",Vantage Outdoor Corp.,"Litware, Inc."
1,PO000941,Delta Sports & Co. Ltd,Carlson Specialties,Delta Sports & Co. Ltd,Carlson Specialties
2,PO000393,Juniper Bikes PLC,West Junction Cycles,Delta Sports & Co. Ltd,West Junction Cycles
3,PO000589,Juniper Components SA,Advanced Bicycles,Delta Sports & Co. Ltd,Advanced Bicycles
4,PO003143,Juniper Bikes PLC,Allenson Cycles,Delta Sports & Co. Ltd,Allenson Cycles


In [77]:
print(f"Linked rows: {len(matched_pairs): ,}")
print(f"Distinct PO references: {matched_pairs['po_reference'].nunique():,}")
print("PO references linked to multiple purchase-orders: ", matched_pairs["po_reference"].value_counts().gt(1).sum())
print("Purchase-order records with a repeated number: ", purchase_orders["number"].duplicated(keep = False).sum())

Linked rows:  476
Distinct PO references: 476
PO references linked to multiple purchase-orders:  0
Purchase-order records with a repeated number:  0


In [78]:
same_buyer = (matched_pairs["company_name_grn"] == matched_pairs["company_name_po"])

same_supplier = (matched_pairs["counterparty_name_grn"] == matched_pairs["counterparty_name_po"])

print(f"Linked records: {len(matched_pairs): ,}")
print(f"Buyer name agrees: {same_buyer.sum(): ,}")
print(f"Supplier name agrees: {same_supplier.sum(): ,}")
print(f"Both names agree: {(same_buyer & same_supplier).sum(): ,}")

Linked records:  476
Buyer name agrees:  151
Supplier name agrees:  476
Both names agree:  151


In [79]:
consistent_pairs = matched_pairs.loc[same_buyer & same_supplier].copy()

display(consistent_pairs[["po_reference", "issue_date_grn", "issue_date_po", "items_grn", "items_po"]].head(5))

,po_reference,issue_date_grn,issue_date_po,items_grn,items_po
0,VO-PO-3378,2025-07-01,2025-06-19,"[[""Adjustable Race"", 3]]","[[""Adjustable Race"", 3]]"
1,PO000941,2024-09-27,2024-09-17,"[[""Paint - Black"", 60], [""Paint - Red"", 60], [...","[[""Paint - Red"", 60], [""Paint - Silver"", 60], ..."
7,PO-2025-2681,2025-05-02,2025-04-20,"[[""Pinch Bolt"", 3]]","[[""Pinch Bolt"", 3]]"
9,PO002579,2025-04-19,2025-04-10,"[[""Chainring Bolts"", 3], [""Chainring Nut"", 3],...","[[""Chainring Bolts"", 3], [""Chainring Nut"", 3],..."
12,PO-2023-86,2023-02-17,2023-02-08,"[[""LL Crankarm"", 550], [""ML Crankarm"", 550], [...","[[""LL Crankarm"", 550], [""ML Crankarm"", 550], [..."


In [80]:
def norm_items(s):
    return sorted((str(d), str(q)) for d, q in json.loads(s))

In [81]:
same_items = consistent_pairs["items_grn"].apply(norm_items) == consistent_pairs["items_po"].apply(norm_items)

grn_dates = pd.to_datetime(consistent_pairs["issue_date_grn"], errors="coerce")
po_dates = pd.to_datetime(consistent_pairs["issue_date_po"], errors="coerce")
grn_after_po = grn_dates >= po_dates

print(f"Pairs with identical item lists: {same_items.sum():,} of {len(consistent_pairs):,}")
print(f"Goods-received note dated on/after PO: {grn_after_po.sum():,} of {len(consistent_pairs):,}")
print(f"Both checks pass: {(same_items & grn_after_po).sum():,} of {len(consistent_pairs):,}")

Pairs with identical item lists: 142 of 151
Goods-received note dated on/after PO: 151 of 151
Both checks pass: 142 of 151


In [82]:
different_items = consistent_pairs.loc[~same_items]

for _, row in different_items.iterrows():
    print(f"\nPO reference: {row['po_reference']}")
    print(f"Ordered items: {row['items_po']}")
    print(f"Received items: {row['items_grn']}")


PO reference: PO000941
Ordered items: [["Paint - Red", 60], ["Paint - Silver", 60], ["Paint - Blue", 60]]
Received items: [["Paint - Black", 60], ["Paint - Red", 60], ["Paint - Silver", 60], ["Paint - Blue", 60]]

PO reference: PO-2025-1697
Ordered items: [["Hex Nut 3", 3], ["Hex Nut 14", 3], ["Hex Nut 15", 3], ["Hex Nut 4", 3], ["Hex Nut 18", 3], ["Hex Nut 19", 3]]
Received items: [["Hex Nut 3", 3], ["Hex Nut 14", 3], ["Hex Nut 4", 3], ["Hex Nut 18", 3], ["Hex Nut 19", 3]]

PO reference: PO000621
Ordered items: [["Metal Bar 2", 550], ["Metal Plate 2", 550], ["Metal Plate 1", 550], ["Metal Plate 3", 550]]
Received items: [["Metal Bar 2", 550], ["Metal Plate 2", 550], ["Metal Plate 3", 550]]

PO reference: PO001508
Ordered items: [["Flat Washer 1", 3], ["Flat Washer 6", 3], ["Flat Washer 2", 3], ["Flat Washer 9", 3], ["Flat Washer 4", 3]]
Received items: [["Flat Washer 1", 3], ["Flat Washer 2", 3], ["Flat Washer 9", 3]]

PO reference: PO000177
Ordered items: [["LL Mountain Seat/Saddle"

In [83]:
shipping_orders = pd.DataFrame(tqdm(light_scan("shipping_order"), desc="Indexing shipping orders"))

shipping_orders.to_parquet("index_shipping_order.parquet", index=False)

invoice_order_ids = inv["refs"].apply(lambda s: json.loads(s).get("order_id"))
shipping_order_ids = shipping_orders["refs"].apply(lambda s: json.loads(s).get("order_id"))

invoice_has_order_id = invoice_order_ids.notna()
shipping_order_id_set = set(shipping_order_ids.dropna())
invoice_order_matches = invoice_order_ids[invoice_has_order_id].isin(shipping_order_id_set)

print(f"Training shipping orders indexed: {len(shipping_orders):,}")
print(f"Invoices with an order ID: {invoice_has_order_id.sum():,}")
print("Invoices whose order ID appears in a shipping order:",f"{invoice_order_matches.sum():,}")

Indexing shipping orders: 0it [00:00, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Training shipping orders indexed: 35,182
Invoices with an order ID: 24,828
Invoices whose order ID appears in a shipping order: 18,166


In [84]:
invoice_for_link = inv.copy()
invoice_for_link["order_id"] = invoice_order_ids

shipping_for_link = shipping_orders.copy()
shipping_for_link["order_id"] = shipping_order_ids

invoice_shipping_pairs = invoice_for_link.dropna(subset=["order_id"]).merge(
    shipping_for_link.dropna(subset=["order_id"]), on="order_id", suffixes=("_invoice", "_shipping"))

same_company = (invoice_shipping_pairs["company_name_invoice"] == invoice_shipping_pairs["company_name_shipping"])
same_counterparty = (invoice_shipping_pairs["counterparty_name_invoice"] == invoice_shipping_pairs["counterparty_name_shipping"])

print(f"Linked record pairs: {len(invoice_shipping_pairs):,}")
print(f"Distinct order IDs: {invoice_shipping_pairs['order_id'].nunique():,}")
print(f"Company names agree: {same_company.sum():,}")
print(f"Counterparty names agree: {same_counterparty.sum():,}")
print(f"Both names agree: {(same_company & same_counterparty).sum():,}")

Linked record pairs: 18,166
Distinct order IDs: 18,166
Company names agree: 1,689
Counterparty names agree: 12,300
Both names agree: 1,150


In [85]:
swapped_company = (invoice_shipping_pairs["company_name_invoice"] == invoice_shipping_pairs["counterparty_name_shipping"])
swapped_counterparty = (invoice_shipping_pairs["counterparty_name_invoice"] == invoice_shipping_pairs["company_name_shipping"])

print(f"Both party names agree when reversed: {(swapped_company & swapped_counterparty).sum():,}")

Both party names agree when reversed: 0


In [86]:
party_consistent_pairs = invoice_shipping_pairs.loc[same_company & same_counterparty].copy()

same_items = party_consistent_pairs["items_invoice"].apply(norm_items) == party_consistent_pairs["items_shipping"].apply(norm_items)

invoice_dates = pd.to_datetime(party_consistent_pairs["issue_date_invoice"], errors="coerce")
shipping_dates = pd.to_datetime(party_consistent_pairs["issue_date_shipping"], errors="coerce")

print(f"Pairs with both party names agreeing: {len(party_consistent_pairs):,}")
print(f"Pairs with identical item lists: {same_items.sum():,}")
print(f"Pairs with dates available on both documents: {(invoice_dates.notna() & shipping_dates.notna()).sum():,}")

Pairs with both party names agreeing: 1,150
Pairs with identical item lists: 1,032
Pairs with dates available on both documents: 1,150


In [87]:
from collections import Counter

def diff_kind(a, b):
    ca = Counter((str(d), str(q)) for d, q in json.loads(a))
    cb = Counter((str(d), str(q)) for d, q in json.loads(b))
    if ca == cb:
        return "identical"
    only_a, only_b = ca - cb, cb - ca
    if {d for d, _ in only_a} & {d for d, _ in only_b}:
        return "quantity differs"
    if only_a and not only_b:
        return "shipping order is missing lines"
    if only_b and not only_a:
        return "shipping order has extra lines"
    return "both sides differ"

kinds = party_consistent_pairs.apply(
    lambda r: diff_kind(r["items_invoice"], r["items_shipping"]), axis=1)
print(kinds.value_counts())

identical                          1032
both sides differ                    52
shipping order has extra lines       36
shipping order is missing lines      30
Name: count, dtype: int64


In [88]:
import os

def load_index(t, split="train"):
    path = f"index_{t}.parquet" if split == "train" else f"index_{t}_{split}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    df = pd.DataFrame(tqdm(light_scan(t, split), desc=f"{t}/{split}"))
    df.to_parquet(path, index=False)
    return df

def get_key(df, key):
    return df["number"] if key == "number" else df["refs"].apply(lambda s: json.loads(s).get(key))

def link_stats(left, right, left_key, right_key, split):
    L, R = load_index(left, split), load_index(right, split)
    L = L.assign(link=get_key(L, left_key)).dropna(subset=["link"])
    R = R.assign(link=get_key(R, right_key)).dropna(subset=["link"])
    p = L.merge(R, on="link", suffixes=("_l", "_r"))
    parties = (p.company_name_l == p.company_name_r) & (p.counterparty_name_l == p.counterparty_name_r)
    items = p.items_l.apply(norm_items) == p.items_r.apply(norm_items)
    return {"pair": f"{left} -> {right}", "split": split, "key_matches": len(p),
            "parties_agree": int(parties.sum()), "parties_and_items_agree": int((parties & items).sum())}

PAIRS = [
    ("invoice", "shipping_order", "order_id", "order_id"),
    ("invoice", "packing_slip", "order_id", "order_id"),
    ("packing_slip", "shipping_order", "order_id", "order_id"),
    ("goods_received_note", "purchase_order", "po_reference", "number"),
    ("credit_note", "invoice", "original_invoice", "number"),
]
results_df = pd.DataFrame([link_stats(*p, s) for p in PAIRS for s in ("train", "validation", "test")])
results_df["party_agree_rate"] = (results_df.parties_agree / results_df.key_matches).round(3)
results_df.to_csv("link_results.csv", index=False)
results_df

,pair,split,key_matches,parties_agree,parties_and_items_agree,party_agree_rate
0,invoice -> shipping_order,train,18166,1150,1032,0.063
1,invoice -> shipping_order,validation,559,182,165,0.326
2,invoice -> shipping_order,test,587,201,187,0.342
3,invoice -> packing_slip,train,12635,1117,747,0.088
4,invoice -> packing_slip,validation,437,228,133,0.522
5,invoice -> packing_slip,test,399,184,118,0.461
6,packing_slip -> shipping_order,train,18086,1116,1004,0.062
7,packing_slip -> shipping_order,validation,611,210,191,0.344
8,packing_slip -> shipping_order,test,604,211,179,0.349
9,goods_received_note -> purchase_order,train,476,151,142,0.317


In [89]:
cn["original_invoice"] = cn["refs"].apply(lambda s: json.loads(s).get("original_invoice"))
m = cn.merge(inv, left_on="original_invoice", right_on="number", suffixes=("_cn", "_inv"))
ok = (m.company_name_cn == m.company_name_inv) & (m.counterparty_name_cn == m.counterparty_name_inv)
print(f"Credit notes with an existing invoice: {len(m):,}; parties agree: {ok.sum():,}")

Credit notes with an existing invoice: 4,623; parties agree: 2,346


In [90]:
inv, shp = load_index("invoice"), load_index("shipping_order")
inv["order_id"], shp["order_id"] = get_key(inv, "order_id"), get_key(shp, "order_id")
p = inv.dropna(subset=["order_id"]).merge(shp.dropna(subset=["order_id"]), on="order_id", suffixes=("_inv", "_shp"))
sub = p[(p.company_name_inv != p.company_name_shp) & (p.counterparty_name_inv == p.counterparty_name_shp)]
same = sub.items_inv.apply(norm_items) == sub.items_shp.apply(norm_items)
print(f"company differs, customer same: {len(sub):,}; identical items: {same.sum():,}")

company differs, customer same: 11,150; identical items: 10,004


In [91]:
# variant/source are covered by the full scan in the source/variant cell; this cell only shows the options field
for t in ["goods_received_note", "invoice", "shipping_order", "credit_note"]:
    ds = load_dataset(dataset_id, t, split="train", streaming=True, columns=["doc_id", "options"])
    first = next(iter(ds))
    print(t, "options example:", first["options"][:300])

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

goods_received_note options example: {"show_discount": true, "show_tax_column": false, "show_notes": true, "show_contact": true, "show_bank": true, "show_ship_to": true, "show_signature": false, "show_item_details": true, "tax_mode": "none", "date_format": "{mm}/{dd}/{yyyy}", "max_items": 5}


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

...

shipping_order options example: {"show_discount": true, "show_tax_column": false, "show_notes": true, "show_contact": true, "show_bank": false, "show_ship_to": true, "show_signature": false, "show_item_details": false, "tax_mode": "given", "date_format": "{d} {month} {yyyy}", "max_items": 20, "show_po_ref": false, "show_salesperso


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

credit_note options example: {"show_discount": true, "show_tax_column": false, "show_notes": true, "show_contact": true, "show_bank": true, "show_ship_to": false, "show_signature": true, "show_item_details": true, "tax_mode": "given", "date_format": "{d} {mon} {yyyy}", "max_items": 5, "show_po_ref": false, "show_salesperson": f


In [92]:
targets = ["VO-22044309", "VO-22045065"]
splits_ = ["train", "validation", "test"]
rows = []
for s in splits_:
    idx = load_index("invoice", s)
    for t in targets:
        rows.append({"invoice": t, "split": s, "found": int((idx["number"] == t).sum())})
pd.DataFrame(rows).pivot(index="invoice", columns="split", values="found")

split,test,train,validation
invoice,,,
VO-22044309,0,0,0
VO-22045065,0,0,0


In [93]:
def agree_report(df, a, b):
    return {
        "n": len(df),
        "same_date":  int((df[f"issue_date_{a}"] == df[f"issue_date_{b}"]).sum()),
        "same_total": int((df[f"total_{a}"] == df[f"total_{b}"]).sum()),
        "same_items": int((df[f"items_{a}"].apply(norm_items) == df[f"items_{b}"].apply(norm_items)).sum()),
    }

trusted = p[(p.company_name_inv == p.company_name_shp) & (p.counterparty_name_inv == p.counterparty_name_shp)]
print("trusted       ", agree_report(trusted, "inv", "shp"))
print("cross-company ", agree_report(sub, "inv", "shp"))

print(pd.crosstab(sub.company_name_inv, sub.company_name_shp).shape)

trusted        {'n': 1150, 'same_date': 1150, 'same_total': 0, 'same_items': 1032}
cross-company  {'n': 11150, 'same_date': 11150, 'same_total': 0, 'same_items': 10004}
(22, 22)


In [94]:
print("share of shipping orders with no total:", shp["total"].isna().mean())
display(shp["total"].value_counts().head())
display(trusted[["total_inv", "total_shp"]].head())

share of shipping orders with no total: 0.0


total
89.46    548
82.30    416
54.54    268
0.12     218
1.75     216
Name: count, dtype: int64

,total_inv,total_shp
2,2624.35,59.37
3,2089.03,47.27
4,2483.90,56.19
28,1485.34,40.54
29,80.28,1.81


In [95]:
grn_po_cross = matched_pairs[(matched_pairs.company_name_grn != matched_pairs.company_name_po)
                             & (matched_pairs.counterparty_name_grn == matched_pairs.counterparty_name_po)]
same = grn_po_cross.items_grn.apply(norm_items) == grn_po_cross.items_po.apply(norm_items)
print(f"GRN/PO company differs, supplier same: {len(grn_po_cross):,}; identical items: {same.sum():,}")

GRN/PO company differs, supplier same: 325; identical items: 312


In [96]:
inv_all = pd.concat([load_index("invoice", s).assign(split=s) for s in splits_], ignore_index=True)
cn_all  = pd.concat([load_index("credit_note", s).assign(split=s) for s in splits_], ignore_index=True)
cn_all["original_invoice"] = get_key(cn_all, "original_invoice")

print("duplicate invoice numbers across splits:", inv_all["number"].duplicated().sum())
inv_split = inv_all.drop_duplicates("number").set_index("number")["split"]
cn_all["invoice_found_in"] = cn_all["original_invoice"].map(inv_split).fillna("not found")
pd.crosstab(cn_all["split"], cn_all["invoice_found_in"], margins=True)

duplicate invoice numbers across splits: 0


invoice_found_in,not found,test,train,validation,All
split,,,,,
test,5399,418,486,0,6303
train,29469,475,4623,540,35107
validation,5434,0,486,417,6337
All,40302,893,5595,957,47747


In [97]:
splits_ = ["train", "validation", "test"]

po_all  = pd.concat([load_index("purchase_order", s).assign(split=s) for s in splits_], ignore_index=True)
grn_all = pd.concat([load_index("goods_received_note", s).assign(split=s) for s in splits_], ignore_index=True)
grn_all["po_reference"] = get_key(grn_all, "po_reference")

print("duplicate PO numbers across splits:", po_all["number"].duplicated().sum())

po_split = po_all.drop_duplicates("number").set_index("number")["split"]
grn_all["po_found_in"] = grn_all["po_reference"].map(po_split).fillna("not found")

counts = pd.crosstab(grn_all["split"], grn_all["po_found_in"], margins=True)
display(counts)

rate = (grn_all["po_found_in"] != "not found").groupby(grn_all["split"]).agg(["sum", "size", "mean"])
rate.columns = ["resolved", "total", "rate"]
rate["rate"] = rate["rate"].round(3)
display(rate)

duplicate PO numbers across splits: 0


po_found_in,not found,test,train,validation,All
split,,,,,
test,341,38,119,12,510
train,2019,106,476,66,2667
validation,394,11,69,38,512
All,2754,155,664,116,3689


,resolved,total,rate
split,,,
test,169,510,0.331
train,648,2667,0.243
validation,118,512,0.230


In [98]:
def meta_scan(t, split):
    path = f"meta_{t}_{split}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    ds = load_dataset(dataset_id, t, split=split, streaming=True, columns=["doc_id", "source", "variant"])
    df = pd.DataFrame(tqdm(ds, desc=f"meta {t}/{split}"))
    df.to_parquet(path, index=False)
    return df

meta_types = ["invoice", "shipping_order", "packing_slip", "credit_note", "goods_received_note", "purchase_order"]
meta = {(t, s): meta_scan(t, s) for t in meta_types for s in splits_}
for (t, s), df in meta.items():
    print(t, s, df["source"].value_counts().to_dict())

src = meta[("invoice", "train")][["doc_id", "source"]].rename(columns={"doc_id": "doc_id_inv", "source": "source_inv"})
q = p.merge(src, on="doc_id_inv")
q["parties_agree"] = (q.company_name_inv == q.company_name_shp) & (q.counterparty_name_inv == q.counterparty_name_shp)
q.groupby("source_inv")["parties_agree"].agg(["size", "mean"])

invoice train {'adventureworks': 23048, 'northwind': 11975, 'chinook': 319}
invoice validation {'adventureworks': 4205, 'northwind': 2197, 'chinook': 44}
invoice test {'adventureworks': 4211, 'northwind': 2110, 'chinook': 49}
shipping_order train {'adventureworks': 23176, 'northwind': 12006}
shipping_order validation {'adventureworks': 4178, 'northwind': 2096}
shipping_order test {'adventureworks': 4111, 'northwind': 2180}
packing_slip train {'adventureworks': 23088, 'northwind': 11952}
packing_slip validation {'adventureworks': 4257, 'northwind': 2199}
packing_slip test {'adventureworks': 4113, 'northwind': 2131}
credit_note train {'adventureworks': 23143, 'northwind': 11964}
credit_note validation {'adventureworks': 4169, 'northwind': 2168}
credit_note test {'adventureworks': 4153, 'northwind': 2150}
goods_received_note train {'adventureworks': 2667}
goods_received_note validation {'adventureworks': 512}
goods_received_note test {'adventureworks': 510}
purchase_order train {'northwin

,size,mean
source_inv,,
adventureworks,11931,0.093035
northwind,6235,0.006415


In [99]:
q["company_agree"] = q.company_name_inv == q.company_name_shp
q["cp_agree"] = q.counterparty_name_inv == q.counterparty_name_shp
q.groupby("source_inv")[["company_agree", "cp_agree", "parties_agree"]].mean().round(3)

,company_agree,cp_agree,parties_agree
source_inv,,,
adventureworks,0.093,1.000,0.093
northwind,0.093,0.059,0.006


In [100]:
import re, unicodedata
SUFFIX = r"\b(inc|incorporated|corp|corporation|co|company|ltd|limited|llc|plc|sa|gmbh|ag)\b"

def norm_name(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(SUFFIX, " ", s)
    return " ".join(s.split())

exact = (p.company_name_inv == p.company_name_shp) & (p.counterparty_name_inv == p.counterparty_name_shp)
norm  = (p.company_name_inv.map(norm_name) == p.company_name_shp.map(norm_name)) & \
        (p.counterparty_name_inv.map(norm_name) == p.counterparty_name_shp.map(norm_name))
print(f"exact: {exact.sum():,}   normalized: {norm.sum():,}   of {len(p):,}")

exact: 1,150   normalized: 1,330   of 18,166


In [101]:
cols = ["company_name_inv", "company_name_shp", "counterparty_name_inv", "counterparty_name_shp"]
p[norm & ~exact][["company_name_inv", "company_name_shp"]].value_counts()

company_name_inv       company_name_shp     
Vantage Outdoor SA     Vantage Outdoor Corp.    93
Vantage Outdoor Corp.  Vantage Outdoor SA       83
Delta Commerce SARL    Delta Commerce SARL       2
Summit Partners Ltd    Summit Partners Ltd       2
Name: count, dtype: int64

In [102]:
def link_stats_x(left, right, lk, rk, ls, rs):
    L, R = load_index(left, ls), load_index(right, rs)
    L = L.assign(link=get_key(L, lk)).dropna(subset=["link"])
    R = R.assign(link=get_key(R, rk)).dropna(subset=["link"])
    m = L.merge(R, on="link", suffixes=("_l", "_r"))
    ok = (m.company_name_l == m.company_name_r) & (m.counterparty_name_l == m.counterparty_name_r)
    return {"left_split": ls, "right_split": rs, "key_matches": len(m), "parties_agree": int(ok.sum())}

display(pd.DataFrame([link_stats_x("invoice", "shipping_order", "order_id", "order_id", a, b)
                      for a, b in [("validation", "train"), ("train", "validation"),
                                   ("test", "train"), ("train", "test")]]))

p["parties_agree"] = (p.company_name_inv == p.company_name_shp) & (p.counterparty_name_inv == p.counterparty_name_shp)
p.groupby("company_name_inv")["parties_agree"].agg(["size", "mean"]).sort_values("size", ascending=False).head(15)

,left_split,right_split,key_matches,parties_agree
0,validation,train,3304,0
1,train,validation,3201,0
2,test,train,3276,0
3,train,test,3238,0


,size,mean
company_name_inv,,
Iris Wheelworks LLC,1118,0.081395
Iris Sports Corp.,1115,0.098655
Delta Sports & Co. Ltd,1113,0.097934
Delta Components SPRL,1100,0.084545
Keystone Components SRL,1092,0.089744
Juniper Components SA,1091,0.072411
Vantage Outdoor SA,1086,0.103131
Harbor Velo Co.,1073,0.100652
Juniper Bikes PLC,1072,0.103545


In [103]:
m = cn_all.merge(inv_all, left_on="original_invoice", right_on="number", suffixes=("_cn", "_inv"))
m["parties_agree"] = (m.company_name_cn == m.company_name_inv) & (m.counterparty_name_cn == m.counterparty_name_inv)
display(m.groupby(["split_cn", "split_inv"])["parties_agree"].agg(["size", "sum"]))

g = grn_all.merge(po_all, left_on="po_reference", right_on="number", suffixes=("_grn", "_po"))
g["parties_agree"] = (g.company_name_grn == g.company_name_po) & (g.counterparty_name_grn == g.counterparty_name_po)
display(g.groupby(["split_grn", "split_po"])["parties_agree"].agg(["size", "sum"]))

size   sum
split_cn   split_inv             
test       test         418   418
           train        486     0
train      test         475     0
           train       4623  2346
           validation   540     0
validation train        486     0
           validation   417   417

size  sum
split_grn  split_po             
test       test          38   38
           train        119    0
           validation    12    0
train      test         106    0
           train        476  151
           validation    66    0
validation test          11    0
           train         69    0
           validation    38   38

In [104]:
cross = [(a, b) for a in splits_ for b in splits_ if a != b]
rows = []
for left, right, lk, rk in PAIRS:
    for ls, rs in cross:
        rows.append({"pair": f"{left} -> {right}", **link_stats_x(left, right, lk, rk, ls, rs)})
cross_df = pd.DataFrame(rows)
cross_df.to_csv("cross_split_results.csv", index=False)

display(cross_df.groupby("pair")[["key_matches", "parties_agree"]].sum())

display(cross_df[cross_df.parties_agree > 0])

,key_matches,parties_agree
pair,,
credit_note -> invoice,1987,0
goods_received_note -> purchase_order,383,0
invoice -> packing_slip,9912,0
invoice -> shipping_order,14215,0
packing_slip -> shipping_order,14047,0


,pair,left_split,right_split,key_matches,parties_agree


# Exploration summary: `AyoubChLin/Company-document-dataset-v2`

## 1. Goal

Check whether this dataset can support the **Cross-Document Validation** part of the project. Only two things were needed:

1. What document types exist, and what each type's schema looks like.
2. Whether documents belonging to the same transaction exist, so they can be compared.

**Short answer:** yes to both, with caveats. Linked documents exist and can be joined on explicit reference fields, but raw key matches overcount consistent pairs (issuer and customer often differ), so only party-consistent pairs should be trusted (Section 5). Party agreement depends strongly on the data source (Section 6, finding 1). The dataset contains ordinary business documents, not customs paperwork (Section 8).

## 2. Dataset overview

- **353,580 documents** in 13 document types, plus an extra `scans` config. Each config has train / validation / test splits (259,497 / 47,114 / 46,969).
- **Per-document fields:** the PDF; flat metadata (`doc_id`, `document_type`, `source`, `variant`, `company*`, `number`, `issue_date`, `currency`, `total`, `items_count`, `counterparty_*`, `layout`, `theme`, `locale`, `pages`); the gold-label JSON `extracted_data`; `file_content`; and render data (`options`, `rendered`, `boxes`, `words`).
- The train counts were obtained by scanning the full train split and then **cross-checked** against the split sizes in the dataset metadata (the split-size metadata cell). They match.

| Document type | Train | Validation | Test | Total |
|---|---:|---:|---:|---:|
| account_statement | 522 | 99 | 107 | 728 |
| credit_note | 35,107 | 6,337 | 6,303 | 47,747 |
| employment_certificate | 202 | 43 | 45 | 290 |
| goods_received_note | 2,667 | 512 | 510 | 3,689 |
| inventory_report | 35 | 6 | 10 | 51 |
| invoice | 35,342 | 6,446 | 6,370 | 48,158 |
| packing_slip | 35,040 | 6,456 | 6,244 | 47,740 |
| payslip | 231 | 30 | 29 | 290 |
| purchase_order | 36,908 | 6,598 | 6,807 | 50,313 |
| quote | 35,004 | 6,313 | 6,429 | 47,746 |
| receipt | 12,105 | 2,218 | 2,133 | 16,456 |
| shipping_order | 35,182 | 6,274 | 6,291 | 47,747 |
| work_order | 31,152 | 5,782 | 5,691 | 42,625 |

The distribution is very uneven. Seven transactional types have 31k to 37k train documents each, while account statements, employment certificates, payslips and inventory reports have only dozens to hundreds.

## 3. Schema

Schema was derived by flattening `extracted_data` for the first 200 train documents of each type (full field lists: `schema_per_type.csv`).

Every type's JSON shares an envelope of `doc_type`, `number`, `issue_date`, `currency` and `parties` (issuer plus customer/supplier, with address, tax ID, bank details). Beyond that, the structure depends on the type:

| Part | Present in |
|---|---|
| `items` (description, quantity, unit_price, discount_rate, tax_rate, line_total, `extra`) | all except employment_certificate and work_order |
| `refs` (cross-references and identifiers) | all except employment_certificate and work_order |
| `totals` | credit_note, invoice, purchase_order, quote, receipt |
| `summary` | account_statement, goods_received_note, inventory_report, packing_slip, payslip, shipping_order, work_order |
| `dates` | goods_received_note, invoice, purchase_order, quote, shipping_order, work_order |
| `notes` | credit_note, invoice, packing_slip, quote, receipt, shipping_order |
| `body` | employment_certificate only |

The `refs` object is where cross-document links live. Keys per type (from the sample; some keys are optional):

| Type | `refs` keys |
|---|---|
| account_statement | account_number, period |
| credit_note | **original_invoice**, customer_id |
| goods_received_note | **po_reference**, carrier |
| inventory_report | scope |
| invoice | **order_id** (134/200), customer_id, po_number (75/200), salesperson (15/200) |
| packing_slip | **order_id** (141/200), customer_id, po_number, salesperson, carrier, tracking_number (32/200) |
| payslip | pay_period |
| purchase_order | account_number, requested_by, carrier |
| quote | customer_id, po_number, salesperson |
| receipt | time, transaction, served_by |
| shipping_order | **order_id**, tracking_number (32/200) |
| employment_certificate, work_order | none |

## 4. How links were tested

For each candidate pair, one document's reference field was joined to the other's key (a document `number` or a shared `order_id`), **within the same split**. Each type and split was scanned once and cached as parquet, using only light columns (no PDFs):

| Pair | Join |
|---|---|
| invoice ↔ shipping_order | `refs.order_id` = `refs.order_id` |
| invoice ↔ packing_slip | `refs.order_id` = `refs.order_id` |
| packing_slip ↔ shipping_order | `refs.order_id` = `refs.order_id` |
| goods_received_note → purchase_order | `refs.po_reference` = PO `number` |
| credit_note → invoice | `refs.original_invoice` = invoice `number` |

A match is called **party-consistent** when the issuing company and the counterparty are the same on both documents. Names are compared by exact string equality. Item lists were compared as sorted (description, quantity) pairs. The same joins were then repeated across split boundaries, for all six ordered split combinations (Section 5, findings 1, 4 and 5).

## 5. Results (in-split key matches)

| Pair | Split | Key matches | Parties agree | Parties + items agree | Party-agree rate |
|---|---|---:|---:|---:|---:|
| invoice → shipping_order | train | 18,166 | 1,150 | 1,032 | 0.063 |
| | validation | 559 | 182 | 165 | 0.326 |
| | test | 587 | 201 | 187 | 0.342 |
| invoice → packing_slip | train | 12,635 | 1,117 | 747 | 0.088 |
| | validation | 437 | 228 | 133 | 0.522 |
| | test | 399 | 184 | 118 | 0.461 |
| packing_slip → shipping_order | train | 18,086 | 1,116 | 1,004 | 0.062 |
| | validation | 611 | 210 | 191 | 0.344 |
| | test | 604 | 211 | 179 | 0.349 |
| goods_received_note → purchase_order | train | 476 | 151 | 142 | 0.317 |
| | validation | 38 | 38 | 36 | 1.000 |
| | test | 38 | 38 | 34 | 1.000 |
| credit_note → invoice | train | 4,623 | 2,346 | 777 | 0.507 |
| | validation | 417 | 417 | 145 | 1.000 |
| | test | 418 | 418 | 132 | 1.000 |

*Party-agree rate = parties agree ÷ key matches.*

**Usable (party-consistent) pairs across all splits, in-split matches only:**

| Pair | Party-consistent | Items also identical |
|---|---:|---:|
| invoice → shipping_order | 1,533 | 1,384 |
| packing_slip → shipping_order | 1,537 | 1,374 |
| invoice → packing_slip | 1,529 | 998 |
| credit_note → invoice | 3,181 | 1,054 |
| goods_received_note → purchase_order | 227 | 212 |

Key matches across split boundaries were checked for all five pairs in all six ordered split combinations (40,544 matches in total). None is party-consistent, so the in-split totals above are complete:

| Pair | Cross-split key matches | Party-consistent |
|---|---:|---:|
| invoice → shipping_order | 14,215 | 0 |
| packing_slip → shipping_order | 14,047 | 0 |
| invoice → packing_slip | 9,912 | 0 |
| credit_note → invoice | 1,987 | 0 |
| goods_received_note → purchase_order | 383 | 0 |

## 6. Key findings

**1. Raw key matches overcount consistent pairs, and the rate depends on the data source.**
In train, only 1,150 of 18,166 invoice/shipping-order matches have the same issuer and counterparty. The customer agrees in 12,300, the issuing company in only 1,689, and swapping the two party fields explains none of the mismatches (0 pairs). A full scan of the `source` column shows the split by source:

| Source (train invoice → shipping order) | Pairs | Issuer agrees | Customer agrees | Both (party-agree) | Party-consistent (≈) |
|---|---:|---:|---:|---:|---:|
| adventureworks | 11,931 | 9.3% | 100% | 9.3% | 1,110 |
| northwind | 6,235 | 9.3% | 5.9% | 0.6% | 40 |

So about 97% of the trusted pairs come from adventureworks. The ~5,900 pairs that differ in both customer and issuer (18,166 − 12,300 = 5,866) are northwind: there the customer agrees in only 5.9% of pairs. In northwind, `order_id` matches mostly link documents for different customers; the items of those pairs were not examined. Within a source, the party-agree rate is flat across issuing companies (7 to 10% for the adventureworks-style issuers, 0.3 to 1.2% for the northwind-style ones among the 15 largest).

Key matches that cross split boundaries are never party-consistent: 0 of 14,215 invoice → shipping order matches across all six ordered split combinations (13,019 of them between validation/test and train).

**2. Cross-company pairs look like the same record with a different issuer, at least for adventureworks (supported, not proven).**
Of the 11,150 train invoice/shipping pairs with a *different issuing company but the same customer*, 10,004 (89.7%) have identical item lists. That is the same rate as in the trusted group (1,032 of 1,150, 89.7%), and all 11,150 and all 1,150 share the same issue date. So customer, items and date agree and only the issuer differs. Overall issuer agreement is 1,689 of 18,166 (9.3%), about 1 in 11. 22 distinct issuers appear on each side of these pairs (apparently 11 per source, inferred from company names). Issuer agreement is 9.3% in both sources even though customer agreement differs sharply (100% vs 5.9%), which fits an issuer drawn independently of the order. This was not tested directly.
- Reuse of IDs *within* a document type is ruled out: purchase-order numbers have no duplicates, and the 18,166 matches cover 18,166 distinct order IDs.
- The same pattern holds for goods-received notes: all 476 matches share the supplier, only 151 share the buyer, and of the 325 pairs with a different buyer 312 (96%) have identical items (142 of 151, 94%, in the trusted group).
- These pairs are better described as same-source records with an inconsistent issuer field than as "not real transactions". Whether to treat them as valid pairs is a modeling decision.
- Totals were never equal (0 of 1,150 and 0 of 11,150) even though every shipping order carries a `total` value. In the five trusted pairs inspected it is about 2 to 3% of the invoice total (for example 59.37 against 2,624.35), so it looks like a shipping or freight charge rather than the invoice amount. This was not verified beyond those rows, so totals are not usable for this comparison.
**3. Item lists match about 90% of the time where they should match.**
For the 1,150 trusted invoice/shipping-order pairs, 1,032 have identical items. Of the 118 that don't, 52 differ on both sides, 36 have extra lines on the shipping order, 30 are missing lines, and none differ only in quantity. The mix has no single pattern, so these were not labeled as defects. Possible causes are partial shipments and generator noise; this was not investigated further.

**4. Goods-received notes vs purchase orders.**
*Trusted pairs (train, 151):* in 151 of 151 the note is dated on or after the PO. Item lists are identical in 142; in the 9 that differ, 7 notes are missing lines (consistent with partial delivery) and 2 have an extra line.

*Resolution:* only 476 of 2,667 train notes (18%) resolve to a PO in the same split, and 7% in validation and test (38 of 512, 38 of 510). Searching all three splits raises this:

| GRN split | Total | PO in train | in validation | in test | Not found | Resolved |
|---|---:|---:|---:|---:|---:|---:|
| train | 2,667 | 476 | 66 | 106 | 2,019 | 648 (24.3%) |
| validation | 512 | 69 | 38 | 11 | 394 | 118 (23.0%) |
| test | 510 | 119 | 12 | 38 | 341 | 169 (33.1%) |

Overall 935 of 3,689 notes (25.3%) resolve. None of the 383 cross-split links is party-consistent, so they look like reference collisions rather than real receipts. All goods-received notes come from adventureworks, while 33,972 of 36,908 train purchase orders (92%) come from northwind, so most POs cannot link to a note.

**5. Credit notes → invoices.**
Only 4,623 of 35,107 train credit notes (13%) point to an invoice in the train split. Searching all splits:

| Credit-note split | Total | Invoice in train | in validation | in test | Not found | Resolved |
|---|---:|---:|---:|---:|---:|---:|
| train | 35,107 | 4,623 | 540 | 475 | 29,469 | 5,638 (16.1%) |
| validation | 6,337 | 486 | 417 | 0 | 5,434 | 903 (14.2%) |
| test | 6,303 | 486 | 0 | 418 | 5,399 | 904 (14.3%) |

Overall 7,445 of 47,747 (15.6%) resolve, and no invoice number occurs in more than one split. Validation and test credit notes never point to each other's invoices, but both point to train invoices. In-split, 2,346 of the 4,623 train matches are party-consistent, and all validation and test matches are. Item agreement is low (777 / 145 / 132), which is expected because a credit usually covers only some lines. None of the 1,987 cross-split links is party-consistent, so they look like chance reference collisions rather than real credit notes.

**6. Dead ends (documented so they are not retried).**
- Invoice `po_number` against purchase-order `number`: 0 of 14,171 match. The formats differ (`PO-59244`, `PO16008173883` vs `PO-2025-1765`, `VO-PO-249`); these are probably the customer's own PO numbers, a hypothesis not tested. Fuzzy matching was not tried.
- The two invoices listed in the first account statement (`VO-22044309`, `VO-22045065`) were searched in all three invoice splits and found in none. Only one statement was tested.

**7. No ground-truth defect labels were found.**
A full scan of `source` and `variant` for six types (invoice, shipping order, packing slip, credit note, goods-received note, purchase order) shows `variant` is `<type>.<source>`, so it carries no extra information. Sources seen:

| Type | Sources (train) |
|---|---|
| invoice | adventureworks 23,048; northwind 11,975; chinook 319 |
| shipping_order, packing_slip, credit_note | adventureworks, northwind |
| goods_received_note | adventureworks only |
| purchase_order | northwind 33,972; adventureworks 2,936 |

Validation and test have the same sources in similar proportions. Sakila did not appear in any of the six types scanned; the other seven types were not scanned for source. `options` holds rendering switches (`show_discount`, `tax_mode`, `date_format`, `max_items`, …), not discrepancy flags.

## 7. What this means for training a cross-document checker

- About 1,500 trusted invoice ↔ shipping-order pairs (and a similar number for packing slip ↔ shipping order) is enough for evaluation and for fine-tuning. It is small for training from scratch. Goods-received note ↔ purchase order is thin (227 pairs). The trusted pairs are almost all adventureworks (about 97% in train invoice → shipping order), so a checker trained on them sees mostly one source.
- All trusted pairs are naturally consistent, so **negative examples must be created** (for example, perturb a quantity, date, total or party on one document) and reported as synthetic.
- The cross-company pairs (same customer, items and date, different issuer) could be used as extra item/quantity examples only if the issuer fields are ignored, and that should be stated.
- No key match across a split boundary is party-consistent for any of the five pairs (0 of 40,544), so consistent pairs stay within one split. Cross-split links can be ignored when building pairs.
- Differences in the data should not be labeled as defects without reliable ground truth.

## 8. Limitations

- **Not customs data.** The 13 types are domestic sales, purchasing, HR and stock documents. There are no bills of lading, customs declarations or certificates of origin. Whatever holds here is a proxy; real import/export documents need their own rule set or labeled examples.
- **Schema samples are not random.** The schema and reference keys come from the first 200 train rows per type and the `options` example from the first row. Streaming order is not shuffled, so this may not represent the full dataset. `source` and `variant` were scanned in full, but only for six types.
- **Only a subset of fields was compared.** Items were matched on (description, quantity) only. Totals were compared for invoice vs shipping order and never matched (finding 2); prices and taxes were not compared. Dates were compared for goods-received note vs purchase order and for invoice vs shipping order (identical in all 12,300 pairs with matching customer).
- **Exact name matching.** A normalized comparison (lower case, punctuation and company suffixes removed) matched 1,330 train invoice/shipping pairs against 1,150 exact. Of the 180 extra pairs, 176 are "Vantage Outdoor SA" ↔ "Vantage Outdoor Corp.", which appear as separate issuers, so normalization merged different entities there. The other 4 have identical issuer names and differ only in the counterparty name after normalization (not inspected). The suffix list was also incomplete, so exact matching was kept.
- **Train differs from validation/test.** The party-agree rate is much lower in train (6 to 9% for order-based pairs, 51% for credit notes → invoices, 32% for goods-received notes → purchase orders) than in validation/test (33 to 52% for order-based pairs, 100% for the other two). Source mix and issuing company do not explain it. No link that crosses a split boundary is party-consistent (0 of 40,544 across the five pairs), so consistent pairs stay within one split. For credit notes, every inconsistent link involves a train document, while validation and test have none in-split. The cause of the train difference was not established.
- **Small validation/test samples** for goods-received note vs purchase order (38 in-split matches each).
- Five pairs were tested. Other possible links (quote ↔ purchase order, account statement ↔ invoices across the full split) were not.

## 9. Files produced

| File | Content |
|---|---|
| `type_counts_train.csv` | train counts from the full scan |
| `type_counts_all_splits.csv` | counts for all splits from dataset metadata |
| `schema_per_type.csv` | flattened `extracted_data` field paths per type |
| `link_results.csv` | the results table in Section 5 |
| `cross_split_results.csv` | key matches and party agreement for every cross-split combination of the five pairs |
| `index_<type>[_<split>].parquet` | cached light indexes used for the joins |
| `meta_<type>_<split>.parquet` | `doc_id`, `source`, `variant` for six types |